# Model Efficiency Test

In [1]:
!pip install transformers datasets optimum[onnxruntime] onnxruntime memory_profiler pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 94.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 45.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!git clone https://github.com/joms-hub/tagalog-fake-news-detection.git
import os
os.chdir('/kaggle/working/tagalog-fake-news-detection')


Cloning into 'tagalog-fake-news-detection'...
remote: Enumerating objects: 303, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 303 (delta 0), reused 1 (delta 0), pack-reused 300 (from 1)
Receiving objects: 100% (303/303), 5.13 MiB | 20.91 MiB/s, done.
Resolving deltas: 100% (162/162), done.


#### 1. Export Models to ONNX

In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# from optimum.exporters.onnx import export

# # Export DistilBERT
distilbert_id = "jcunado/distilbert-multilingual-fake-news-filipino"
# distilbert_model = AutoModelForSequenceClassification.from_pretrained(distilbert_id)
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_id)
# export(model=distilbert_model, tokenizer=distilbert_tokenizer, output="distilbert.onnx", task="text-classification")

# # Export MobileBERT
mobilebert_id = "jcunado/MobileBERT-tagalog-fake-news"
# mobilebert_model = AutoModelForSequenceClassification.from_pretrained(mobilebert_id)
mobilebert_tokenizer = AutoTokenizer.from_pretrained(mobilebert_id)
# export(model=mobilebert_model, tokenizer=mobilebert_tokenizer, output="mobilebert.onnx", task="text-classification")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#### 2. Load test data

In [4]:
from datasets import load_from_disk

test_dataset = load_from_disk("/kaggle/working/tagalog-fake-news-detection/tokenized/DistilBERT_test")
test_texts = test_dataset["article"]

#### 3. Tokenize and run ONNX inference for latency

In [5]:
import onnxruntime as ort
import time
from memory_profiler import memory_usage
import os
import numpy as np

def benchmark_model(onnx_model_path, tokenizer_id, test_texts, max_length=128):
    session = ort.InferenceSession(onnx_model_path)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
    # Latency
    latencies = []
    for text in test_texts:
        inputs = tokenizer(text, return_tensors="np", truncation=True, padding="max_length", max_length=max_length)
        input_feed = {k: v for k, v in inputs.items()}
        start = time.time()
        _ = session.run(None, input_feed)
        end = time.time()
        latencies.append((end - start) * 1000)
    avg_latency = sum(latencies) / len(latencies)
    # RAM
    def infer_with_memory(text):
        inputs = tokenizer(text, return_tensors="np", truncation=True, padding="max_length", max_length=max_length)
        input_feed = {k: v for k, v in inputs.items()}
        def model_infer():
            session.run(None, input_feed)
        mem_usage = memory_usage(model_infer, max_iterations=1)
        return max(mem_usage)
    ram_usages = [infer_with_memory(text) for text in test_texts]
    peak_ram = max(ram_usages)
    # Model Size
    model_size = os.path.getsize(onnx_model_path) / 1024**2  # MB
    return avg_latency, peak_ram, model_size

#### 4. Benchmark DistilBERT

In [9]:
distilbert_latency, distilbert_ram, distilbert_size = benchmark_model(
    "/kaggle/input/distilbert-onnx-format/model.onnx", distilbert_id, test_texts
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

#### 5. Benchmark MobileBERT

In [10]:
mobilebert_latency, mobilebert_ram, mobilebert_size = benchmark_model(
    "/kaggle/input/mobilebert-onnx-format/model.onnx", mobilebert_id, test_texts
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

#### 6. Ensemble Benchmark

In [11]:
distilbert_session = ort.InferenceSession("/kaggle/input/distilbert-onnx-format/model.onnx")
mobilebert_session = ort.InferenceSession("/kaggle/input/mobilebert-onnx-format/model.onnx")
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_id)
mobilebert_tokenizer = AutoTokenizer.from_pretrained(mobilebert_id)

# Ensemble F1 scores (update as needed)
F1_DistilBERT = 0.9709
F1_MobileBERT = 0.9647
w1 = F1_DistilBERT / (F1_DistilBERT + F1_MobileBERT)
w2 = F1_MobileBERT / (F1_DistilBERT + F1_MobileBERT)

ensemble_latencies = []
ensemble_ram_usages = []

for text in test_texts:
    distilbert_inputs = distilbert_tokenizer(text, return_tensors="np", truncation=True, padding="max_length", max_length=128)
    distilbert_feed = {k: v for k, v in distilbert_inputs.items()}
    mobilebert_inputs = mobilebert_tokenizer(text, return_tensors="np", truncation=True, padding="max_length", max_length=128)
    mobilebert_feed = {k: v for k, v in mobilebert_inputs.items()}

    # Latency
    start = time.time()
    distilbert_logits = distilbert_session.run(None, distilbert_feed)[0]
    mobilebert_logits = mobilebert_session.run(None, mobilebert_feed)[0]
    end = time.time()
    ensemble_latencies.append((end - start) * 1000)

    # RAM usage
    def model_infer():
        distilbert_session.run(None, distilbert_feed)
        mobilebert_session.run(None, mobilebert_feed)
    mem_usage = memory_usage(model_infer, max_iterations=1)
    ensemble_ram_usages.append(max(mem_usage))

avg_ensemble_latency = sum(ensemble_latencies) / len(ensemble_latencies)
peak_ensemble_ram = max(ensemble_ram_usages)
ensemble_model_size = (
    os.path.getsize("/kaggle/input/distilbert-onnx-format/model.onnx") +
    os.path.getsize("/kaggle/input/mobilebert-onnx-format/model.onnx")
) / 1024**2  # MB

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

#### 7. Output Results as Table

In [12]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["DistilBERT", "MobileBERT", "Ensemble"],
    "Inference Latency (ms)": [distilbert_latency, mobilebert_latency, avg_ensemble_latency],
    "Peak RAM Usage (MB)": [distilbert_ram, mobilebert_ram, peak_ensemble_ram],
    "Model Size (MB)": [distilbert_size, mobilebert_size, ensemble_model_size]
})

print(results)

        Model  Inference Latency (ms)  Peak RAM Usage (MB)  Model Size (MB)
0  DistilBERT               66.258428          1444.609375       516.335463
1  MobileBERT               47.953814          1032.605469        94.376520
2    Ensemble              125.965992          1508.613281       610.711983
